In [1]:
!pip install geopy pandas tqdm

  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached geographiclib-2.1-py3-none-any.whl.metadata (1.6 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
Using cached geographiclib-2.1-py3-none-any.whl (40 kB)

   ---------------------------------------- 0/2 [geographiclib]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]




[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import geopandas as gpd
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

In [9]:
gdf = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Santa Maria\Padronização\CVS_PRONTO\ETE.gpkg')

In [10]:
gdf

,id,REGIONAL,UF,UN,MUNICIPIO,BAIRRO,BACIA,NOME,NOTAS,DESCRICAO,...,BLOCO,CREATED_USER,CREATED_DATE,LAST_EDITED_USER,LAST_EDITED_DATE,DISTRITO,SUPERINTENDENCIA,GRUPO_EXECUTIVO,ETAPA_CICLO,geometry
0,NaN,None,None,None,None,None,None,ETE Lorenzi,None,None,...,NaN,None,NaT,None,NaT,None,None,None,Projeto Conceitual,POINT (227560.569 6708203.71)


In [11]:
# ----------------------------
# 1) GARANTIR CRS UTM 22S
#    (se seu gdf já tem crs correto, pode pular o set_crs)
# ----------------------------
# Ex.: SIRGAS2000 / UTM 22S -> EPSG:31982
# Ex.: WGS84 / UTM 22S      -> EPSG:32722
# gdf = ...  # seu GeoDataFrame de pontos
# if gdf.crs is None:
#     gdf = gdf.set_crs(epsg=31982)  # ajuste para 32722 se for o seu caso

# ----------------------------
# 2) REPROJETAR PARA WGS84
# ----------------------------
gdf_wgs = gdf.to_crs(epsg=4326).copy()
gdf_wgs["lat"] = gdf_wgs.geometry.y
gdf_wgs["lon"] = gdf_wgs.geometry.x

# ----------------------------
# 3) CONFIGURAR NOMINATIM + RATE LIMIT
# ----------------------------
geolocator = Nominatim(user_agent="bairro_osm_extractor")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)  # gentil com a API

# ----------------------------
# 4) FUNÇÃO PARA EXTRAIR BAIRRO
# ----------------------------
def extrair_bairro(lat, lon):
    try:
        loc = reverse((lat, lon), language="pt")
        if not loc or "address" not in loc.raw:
            return None
        addr = loc.raw["address"]
        # ordem de preferência dos campos que costumam representar "bairro"
        return (
            addr.get("suburb")
            or addr.get("neighbourhood")
            or addr.get("city_district")
            or addr.get("quarter")
            or addr.get("village")
            or addr.get("town")
            or addr.get("city")
        )
    except Exception:
        return None

# ----------------------------
# 5) CACHE: evita repetir chamadas para as mesmas coords
#    (arredonda p/ reduzir duplicatas quase idênticas)
# ----------------------------
def chave_cache(lat, lon, casas=5):
    return (round(lat, casas), round(lon, casas))

coords = gdf_wgs[["lat", "lon"]].copy()
coords["key"] = coords.apply(lambda r: chave_cache(r["lat"], r["lon"]), axis=1)

# Deduplica
coords_uniq = coords.drop_duplicates("key").reset_index(drop=True)

# Consulta com barra de progresso
tqdm.pandas(desc="Buscando bairros (OSM)")
coords_uniq["bairro"] = coords_uniq.progress_apply(
    lambda r: extrair_bairro(r["lat"], r["lon"]), axis=1
)

# Mapeia de volta para todas as linhas
mapa_bairros = dict(zip(coords_uniq["key"], coords_uniq["bairro"]))
gdf_wgs["bairro"] = coords["key"].map(mapa_bairros)

# ----------------------------
# 6) (Opcional) ANEXAR AO GDF ORIGINAL
# ----------------------------
# Se você quer manter o CRS UTM e apenas adicionar a coluna:
gdf_final = gdf.copy()
gdf_final["BAIRRO"] = gdf_wgs["bairro"]

# Resultado:
# gdf_final tem a sua geometria original (UTM 22S) + coluna 'bairro'
print(gdf_final[["BAIRRO", "geometry"]].head())


Buscando bairros (OSM): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.20it/s]

    BAIRRO                       geometry
0  Lorenzi  POINT (227560.569 6708203.71)


In [7]:
# salva seu GeoDataFrame em um arquivo .gpkg
gdf_final.to_file(r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Santa Maria\Padronização\CVS_PRONTO\ETE.gpkg", driver="GPKG")